# Arabic NLP — Dataset EDA
**Student B: Hiba El Ouazi**

Exploratory Data Analysis for ANERcorp, AQMAR, and QCRI Dialect POS datasets.

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

ROOT = Path('..').resolve().parent
sys.path.insert(0, str(ROOT))
ANALYSIS = ROOT / 'data' / 'analysis'
print('Root:', ROOT)

## 1. ANERcorp — NER Dataset

In [ ]:
with open(ANALYSIS / 'anercorp_stats.json') as f:
    anercorp = json.load(f)

print(f"Sentences : {anercorp['total_sentences']:,}")
print(f"Tokens    : {anercorp['total_tokens']:,}")
print(f"Avg length: {anercorp['avg_sent_length']} tokens")
print(f"NE ratio  : {anercorp['ne_ratio']:.2%}")
print()
print('Entity counts:')
for etype, count in anercorp['entity_counts'].items():
    print(f'  {etype:6s}: {count:,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Entity type distribution
entity_counts = anercorp['entity_counts']
colors = ['#4A90D9', '#E07B54', '#27AE60', '#9B59B6']
axes[0].bar(entity_counts.keys(), entity_counts.values(), color=colors[:len(entity_counts)])
axes[0].set_title('ANERcorp — Entity Type Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, (k, v) in enumerate(entity_counts.items()):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=10)

# Tag distribution (O vs entities)
tag_dist = anercorp['tag_distribution']
o_count = tag_dist.get('O', 0)
ne_count = anercorp['total_tokens'] - o_count
axes[1].pie([o_count, ne_count], labels=['Outside (O)', 'Named Entity'],
            colors=['#ECF0F1', '#7C3AED'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('ANERcorp — Token Label Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig(ROOT / 'data' / 'analysis' / 'anercorp_eda.png', bbox_inches='tight')
plt.show()
print('Saved: data/analysis/anercorp_eda.png')

## 2. AQMAR — NER Dataset

In [ ]:
with open(ANALYSIS / 'aqmar_ner_stats.json') as f:
    aqmar = json.load(f)

print(f"Sentences : {aqmar['total_sentences']:,}")
print(f"Tokens    : {aqmar['total_tokens']:,}")
print(f"Avg length: {aqmar['avg_sent_length']} tokens")
print(f"NE ratio  : {aqmar['ne_ratio']:.2%}")
print()
print('Entity counts (normalized):')
for etype, count in aqmar['entity_counts'].items():
    print(f'  {etype:6s}: {count:,}')

## 3. QCRI Dialect POS

In [ ]:
with open(ANALYSIS / 'arabic_pos_dialect_stats.json') as f:
    pos_stats = json.load(f)

print(f"Total sentences: {pos_stats['total_sentences']:,}")
print(f"Total tokens   : {pos_stats['total_tokens']:,}")
print()
print('Per-dialect breakdown:')
for dialect, stats in pos_stats['per_dialect'].items():
    print(f"  {stats['dialect']:12s}: {stats['sentences']:,} sents, "
          f"{stats['tokens']:,} tokens, {stats['unique_pos']:,} unique POS tags")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sentences per dialect
dialects = [s['dialect'] for s in pos_stats['per_dialect'].values()]
sents    = [s['sentences'] for s in pos_stats['per_dialect'].values()]
colors   = ['#4A90D9', '#27AE60', '#E07B54', '#9B59B6']
axes[0].bar(dialects, sents, color=colors)
axes[0].set_title('QCRI POS — Sentences per Dialect', fontweight='bold')
axes[0].set_ylabel('Sentences')

# Top POS tags across all dialects
all_top = {}
for s in pos_stats['per_dialect'].values():
    for tag, count in s.get('top_pos_tags', {}).items():
        all_top[tag] = all_top.get(tag, 0) + count
top10 = dict(sorted(all_top.items(), key=lambda x: -x[1])[:10])
axes[1].barh(list(top10.keys()), list(top10.values()), color='#7C3AED')
axes[1].set_title('QCRI POS — Top 10 Tags (all dialects)', fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(ROOT / 'data' / 'analysis' / 'pos_eda.png', bbox_inches='tight')
plt.show()
print('Saved: data/analysis/pos_eda.png')

## 4. Combined Dataset Summary

In [ ]:
print('=' * 55)
print('  DATASET SUMMARY')
print('=' * 55)
print(f"  {'Dataset':<20} {'Task':<12} {'Sents':>8} {'Tokens':>10}")
print('  ' + '─' * 51)
print(f"  {'ANERcorp':<20} {'NER':<12} {anercorp['total_sentences']:>8,} {anercorp['total_tokens']:>10,}")
print(f"  {'AQMAR':<20} {'NER':<12} {aqmar['total_sentences']:>8,} {aqmar['total_tokens']:>10,}")
print(f"  {'QCRI Dialect POS':<20} {'POS':<12} {pos_stats['total_sentences']:>8,} {pos_stats['total_tokens']:>10,}")
print(f"  {'WikiCoref':<20} {'Coreference':<12} {'—':>8} {'—':>10} (placeholder)")
print('=' * 55)
total_sents = anercorp['total_sentences'] + aqmar['total_sentences'] + pos_stats['total_sentences']
total_toks  = anercorp['total_tokens'] + aqmar['total_tokens'] + pos_stats['total_tokens']
print(f"  {'TOTAL':<20} {'':12} {total_sents:>8,} {total_toks:>10,}")
print('=' * 55)

## 5. Class Imbalance Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ANERcorp imbalance
tag_dist = anercorp['tag_distribution']
tags   = list(tag_dist.keys())[:10]
counts = [tag_dist[t] for t in tags]
bar_colors = ['#E74C3C' if t == 'O' else '#7C3AED' for t in tags]
axes[0].bar(tags, counts, color=bar_colors)
axes[0].set_title('ANERcorp — Tag Distribution (class imbalance)', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# NE ratio comparison across datasets
datasets = ['ANERcorp', 'AQMAR']
ne_ratios = [anercorp['ne_ratio'] * 100, aqmar['ne_ratio'] * 100]
axes[1].bar(datasets, ne_ratios, color=['#4A90D9', '#27AE60'], width=0.4)
axes[1].set_title('Named Entity Token Ratio per Dataset', fontweight='bold')
axes[1].set_ylabel('NE Token %')
axes[1].set_ylim(0, 30)
for i, v in enumerate(ne_ratios):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(ROOT / 'data' / 'analysis' / 'class_imbalance.png', bbox_inches='tight')
plt.show()
print('Saved: data/analysis/class_imbalance.png')